# Phase 2 — SFT LoRA puis DPO de Qwen3-1.7B-Base

Fine-tuning supervisé (SFT) de `Qwen/Qwen3-1.7B-Base` sur le dataset anonymisé de la phase 1, avec LoRA, puis alignement par préférences (DPO) du modèle SFT.

- **Prérequis** : GPU, fichiers `data/anonymized/sft_*_{train,validation,test}.anonymized.jsonl` produits par `data_prep_MediQAl` (français, réponse unique), `data_prep_SFT_MediQAl_mcqm` (français, réponses multiples) et `data_prep_SFT_UltraMedical` (anglais). Les QCM français contiennent les propositions A à E ; la réponse attendue donne la ou les lettres et leur texte.
- **Prérequis DPO** : fichiers `data/anonymized/dpo_mediqal_mcqm_*` et `dpo_ultramedical_*` produits par `data_prep_DPO_MediQAl_mcqm` et `data_prep_DPO_UltraMedical`.
- **Étapes** : données → modèle → LoRA → entraînement SFT → validation intermédiaire → DPO → validation DPO → modèle final fusionné.
- **Suivi** : MLflow sur DagsHub (`MLFLOW_BACKEND = "dagshub"`), consultable de partout et quelle que soit la machine d'entraînement. En local (`"local"`) : `mlflow ui --backend-store-uri sqlite:///mlflow.db` depuis la racine du projet.

In [1]:
# Installation (Colab ou machine GPU) : décommenter si besoin
# %pip install -q torch transformers peft trl datasets accelerate mlflow dagshub pandas

## 1. Configuration

In [2]:
import functools
import gc
import json
import math
import re
from collections import Counter
from pathlib import Path

import mlflow
import pandas as pd
import torch
from datasets import DatasetDict, concatenate_datasets, load_dataset
from IPython.display import display
from peft import LoraConfig, PeftModel
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import DPOConfig, DPOTrainer, SFTConfig, SFTTrainer

from projet_14.mediqal import answer_letters

MODEL_NAME = "Qwen/Qwen3-1.7B-Base"
# Données SFT de la phase 1, anonymisées : fichier -> langue
# Le SFT couvre les formats et les langues du DPO (section 7) : réponses uniques et listes en français, anglais
SFT_DATASETS = {
    "sft_mediqal_mcqu": "fr",   # QCM à réponse unique, avec cas clinique
    "sft_mediqal_mcqm": "fr",   # QCM à réponses multiples (liste), avec et sans cas clinique
    "sft_ultramedical": "en",   # questions et réponses rédigées en anglais
}

# Suivi MLflow : "dagshub" (serveur distant) ou "local" (fichier ../mlflow.db)
MLFLOW_BACKEND = "dagshub"
DAGSHUB_REPO_OWNER = "obiyohan"
DAGSHUB_REPO_NAME = "my-ml-mlflow"

# Petit run pour valider la pipeline avant l'entraînement complet
SMALL_RUN = False  # True pour un run rapide sur 100 exemples, False pour le run complet

SEED = 42
# Version des données de la phase 1 : dans le nom des runs, pour ne pas écraser les checkpoints d'une version à l'autre
# v1 : propositions absentes du prompt ; v2 : propositions dans le prompt, répartition par cas clinique,
#      DPO anglais équilibré sur la longueur
DATA_VERSION = "v2"
RUN_NAME = f"sft-lora-r16-{DATA_VERSION}-seed{SEED}" + ("-small" if SMALL_RUN else "")
OUTPUT_DIR = Path("../outputs") / RUN_NAME  # checkpoints et adaptateur final

set_seed(SEED)
if torch.cuda.is_available():
    device, bf16 = "cuda", torch.cuda.is_bf16_supported()  # bf16 absent sur T4 : fp16 à la place
elif torch.xpu.is_available():
    device, bf16 = "xpu", torch.xpu.is_bf16_supported()  # GPU Intel Arc
else:
    device, bf16 = "cpu", False
dtype = torch.bfloat16 if bf16 else torch.float16 if device != "cpu" else torch.float32
print(f"device={device}, dtype={dtype}")

if device == "xpu":
    # GPU Intel sous Windows : deux contournements
    # 1. Pas de compilateur C pour Triton : TRL calcule la loss en PyTorch pur
    import trl.kernels.chunked_logprob as chunked_logprob
    chunked_logprob._TRITON_AVAILABLE = False
    # 2. La norme des gradients en mode « foreach » fait planter le GPU : calcul tenseur par tenseur
    torch.nn.utils.clip_grad_norm_ = functools.partial(torch.nn.utils.clip_grad.clip_grad_norm_, foreach=False)

/home/johan/projects/P14/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device=cuda, dtype=torch.bfloat16


## 2. Données : train / validation / test

In [3]:
# Format prompt / completion : la loss ne porte que sur la réponse de l'assistant
def to_prompt_completion(record):
    return {"prompt": record["messages"][:-1], "completion": record["messages"][-1:]}


# Chaque source garde son découpage train / validation / test ; les sources sont ensuite mélangées
parts = {split: [] for split in ("train", "validation", "test")}
for stem, lang in SFT_DATASETS.items():
    files = {split: f"../data/anonymized/{stem}_{split}.anonymized.jsonl" for split in parts}
    ds = load_dataset("json", data_files=files).select_columns(["messages"])
    for split, part in ds.items():
        parts[split].append(part.add_column("source", [stem] * len(part)).add_column("lang", [lang] * len(part)))

splits = DatasetDict({split: concatenate_datasets(p).shuffle(seed=SEED) for split, p in parts.items()})
splits = splits.map(to_prompt_completion, remove_columns=["messages"])

# Petit run : sous-échantillon (toutes sources mélangées)
if SMALL_RUN:
    for name, size in {"train": 50, "validation": 15, "test": 24}.items():
        splits[name] = splits[name].select(range(size))

pd.DataFrame({name: pd.Series(ds["source"]).value_counts() for name, ds in splits.items()}).fillna(0).astype(int)

Map: 100%|██████████| 2180/2180 [00:00<00:00, 11044.53 examples/s]


,train,validation,test
sft_mediqal_mcqm,2400,380,931
sft_mediqal_mcqu,2454,342,749
sft_ultramedical,1500,300,500


## 3. Tokenizer et modèle

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Gabarit ChatML, avec une réponse de l'assistant terminée par <|endoftext|> (eos_token) au lieu de <|im_end|>.
# Dans Qwen3-Base, <|im_start|> et <|im_end|> n'ont jamais été entraînés (embeddings à leur valeur d'initialisation)
# et LoRA ne modifie pas les embeddings : le modèle ne peut pas apprendre à produire <|im_end|>, donc à s'arrêter.
# <|endoftext|> a été appris au pré-entraînement : le modèle sait le prédire, et vLLM s'arrête dessus par défaut.
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{{ '<|im_start|>' + message['role'] + '\n' + message['content'] }}"
    "{% if message['role'] == 'assistant' %}{{ eos_token }}{% else %}{{ '<|im_end|>\n' }}{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
)

example = splits["train"][0]
print(tokenizer.apply_chat_template(example["prompt"] + example["completion"], tokenize=False))

<|im_start|>system
Tu es un assistant médical expert. Réponds de façon précise et concise.<|im_end|>
<|im_start|>user
Un homme de 45 ans, dont le poids normal est de 60 kg, présente des adénopathies cervicales bilatérales, un amaigrissement de 7 kg en 6 mois, l'absence de fièvre, pas de sueurs nocturnes. L'examen permet de percevoir une rate débordant de 4 cm le rebord costal. Pas d'hépatomégalie, pas d'adénopathie dans les autres territoires accessibles à l'examen clinique. Le bilan objective : VS supérieure à 100 à la première heure, fibrinémie à 9 g/l, leucopénie à 3.000 globules blancs, pas de cellule anormale dans le sang, radiographie pulmonaire normale

La lymphographie était négative, et la laparotomie a objectivé une rate de 400 grammes, des ganglions du hile splénique et du pédicule hépatique envahis, un foie et une moelle osseuse histologiquement normaux. Le stade pathologique (PS) de ce patient est :

Propositions :
A. PS IIE
B. PS IIB
C. PS IIIA
D. PS IIIB
E. PS IVB<|im_en

In [5]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype).to(device)
print(f"{model.num_parameters() / 1e9:.2f} B paramètres")

Loading weights: 100%|██████████| 310/310 [00:02<00:00, 123.37it/s]


1.72 B paramètres


## 4. LoRA

In [6]:
# Seules de petites matrices de rang r sont entraînées : le modèle de base reste figé
lora_config = LoraConfig(
    r=16,                         # rang des matrices d'adaptation
    lora_alpha=32,                # facteur d'échelle (souvent 2 × r)
    lora_dropout=0.05,
    target_modules="all-linear",  # couches d'attention et MLP
    task_type="CAUSAL_LM",
)

## 5. Entraînement SFT

In [ ]:
# Suivi MLflow : un run par entraînement, hyperparamètres et métriques enregistrés
if MLFLOW_BACKEND == "dagshub":
    import dagshub

    # Jeton lu dans la variable d'environnement DAGSHUB_USER_TOKEN,
    # sinon connexion via le navigateur au premier lancement (jeton mémorisé ensuite)
    dagshub.init(repo_owner=DAGSHUB_REPO_OWNER, repo_name=DAGSHUB_REPO_NAME, mlflow=True)
else:
    mlflow.set_tracking_uri("sqlite:///../mlflow.db")
print(f"Suivi MLflow : {mlflow.get_tracking_uri()}")

mlflow.set_experiment("qwen3-1.7b-sft-lora")
mlflow.start_run(run_name=RUN_NAME)
mlflow.log_params({"model": MODEL_NAME, "data_version": DATA_VERSION, "small_run": SMALL_RUN, "device": device, "lora_r": lora_config.r, "lora_alpha": lora_config.lora_alpha, "lora_dropout": lora_config.lora_dropout})

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    run_name=RUN_NAME,
    seed=SEED,
    # Optimisation
    num_train_epochs=1,
    max_steps=30 if SMALL_RUN else -1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # batch effectif = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.03,  # 3 % des étapes en montée progressive du learning rate
    max_length=1024,
    gradient_checkpointing=True,  # moins de mémoire GPU, un peu plus lent
    bf16=bf16,
    fp16=dtype == torch.float16,
    # Évaluation et checkpoints
    eval_strategy="steps",
    eval_steps=10 if SMALL_RUN else 100,
    save_steps=10 if SMALL_RUN else 100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    # Logs
    logging_steps=5 if SMALL_RUN else 20,
    report_to="mlflow",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=splits["train"],
    eval_dataset=splits["validation"],
    processing_class=tokenizer,
    peft_config=lora_config,
)
trainer.model.print_trainable_parameters()

Accessing as ObiYohan

Initialized MLflow to track repo "obiyohan/my-ml-mlflow"

Repository obiyohan/my-ml-mlflow initialized!

Suivi MLflow : https://dagshub.com/obiyohan/my-ml-mlflow.mlflow


Truncating train dataset: 100%|██████████| 6354/6354 [00:00<00:00, 6963.84 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 6354/6354 [00:00<00:00, 28309.22 examples/s]
Dropping fully masked examples from eval dataset: 100%|██████████| 1022/1022 [00:00<00:00, 26598.61 examples/s]


trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


In [8]:
# Référence avant entraînement (adaptateur LoRA initialisé à zéro = modèle de base)
val_before = trainer.evaluate(metric_key_prefix="val_before")
val_before

/home/johan/projects/P14/.venv/lib/python3.14/site-packages/triton/knobs.py:504: UserWarning: TRITON_DEFAULT_FP_FUSION is unset: defaulting to FP fusion ENABLED for backward compatibility in Triton 3.9. The default will change to DISABLED in Triton 3.10; set TRITON_DEFAULT_FP_FUSION=1 to keep fusion enabled or =0 to adopt the new default early and silence this warning.
  warn_once(FP_FUSION_DEFAULT_WARNING)


Training Loss,Validation Loss,Step,Before Loss,Entropy,Mean Token Accuracy,Num Tokens
No log,No log,0,0.778958,0.831962,0.775677,0.000000


{'val_before_loss': 0.7789584994316101,
 'eval_entropy': 0.8319617547094822,
 'eval_mean_token_accuracy': 0.7756772311404347,
 'eval_num_tokens': 0.0}

In [9]:
# Reprise après interruption : trainer.train(resume_from_checkpoint=True)
train_result = trainer.train()
train_result.metrics

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
100,0.673220,0.658227,0.658874,0.799466,608440.000000
200,0.668403,0.649556,0.646022,0.801416,1215508.000000
300,0.658481,0.647443,0.636033,0.801815,1814238.000000
400,0.621562,0.646693,0.633423,0.801418,2401060.000000
500,0.559342,0.649311,0.579626,0.801868,3008365.000000
600,0.528169,0.651288,0.562559,0.801807,3608264.000000
700,0.571853,0.650257,0.565505,0.802211,4216363.000000
796,0.581993,0.649968,0.567012,0.801864,4779912.000000


{'train_runtime': 3570.5841,
 'train_samples_per_second': 3.559,
 'train_steps_per_second': 0.223,
 'total_flos': 6.302567052597658e+16,
 'train_loss': 0.6230331018342444,
 'epoch': 2.0}

In [10]:
# Sauvegarde de l'adaptateur LoRA (meilleur checkpoint) et de la configuration du run
FINAL_DIR = OUTPUT_DIR / "final"
trainer.save_model(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))

run_config = {
    "model": MODEL_NAME,
    "seed": SEED,
    "small_run": SMALL_RUN,
    "lora": {"r": lora_config.r, "alpha": lora_config.lora_alpha, "dropout": lora_config.lora_dropout},
    "data": {name: len(ds) for name, ds in splits.items()},
    "training_args": training_args.to_dict(),
}
(FINAL_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2, default=str), encoding="utf-8")

4580

## 6. Validation intermédiaire

1. **Loss, perplexité et token accuracy** sur validation (avant / après SFT) et test.
2. **Qualité des réponses générées** sur des exemples de test, modèle de base vs modèle SFT, sur les **lettres** données dans la réponse : accuracy (exactement les bonnes lettres) et F1 (crédit partiel pour les réponses multiples). Génération par lots pour bien charger le GPU.

La **latence** n'est pas mesurée ici : avec l'adaptateur LoRA non fusionné et Transformers, elle ne serait pas représentative. Elle sera mesurée en phase 3, sur le modèle final fusionné servi par vLLM.

In [11]:
val_after = trainer.evaluate(metric_key_prefix="val_after")
test_metrics = trainer.evaluate(splits["test"], metric_key_prefix="test")

# token_accuracy : part des tokens de la réponse attendue que le modèle prédit correctement
# (TRL la nomme « eval_mean_token_accuracy » quel que soit le préfixe : on la renomme ici)
evaluations = {"val_before": val_before, "val_after": val_after, "test": test_metrics}
summary = pd.DataFrame({
    name: {"loss": m[f"{name}_loss"], "token_accuracy": m["eval_mean_token_accuracy"]}
    for name, m in evaluations.items()
}).T
summary["perplexity"] = summary["loss"].map(math.exp)

# Même mesure sur le test, source par source : chaque format doit être appris
per_source = {}
for source in SFT_DATASETS:
    subset = splits["test"].filter(lambda example: example["source"] == source)
    if len(subset):
        m = trainer.evaluate(subset, metric_key_prefix=f"test_{source}")
        per_source[f"test_{source}"] = {"loss": m[f"test_{source}_loss"], "token_accuracy": m["eval_mean_token_accuracy"]}
per_source_summary = pd.DataFrame(per_source).T
per_source_summary["perplexity"] = per_source_summary["loss"].map(math.exp)
pd.concat([summary, per_source_summary])

Training Loss,Validation Loss,Step,After Loss,Entropy,Mean Token Accuracy,Num Tokens
0.581993,No log,796,0.646693,0.633423,0.801418,4779912.000000


Dropping fully masked examples from eval dataset: 100%|██████████| 2180/2180 [00:00<00:00, 29811.04 examples/s]


Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
0.581993,0.592630,796,0.582508,0.815342,4779912.000000


Dropping fully masked examples from eval dataset: 100%|██████████| 749/749 [00:00<00:00, 20038.74 examples/s]


Training Loss,Validation Loss,Step,Sft Mediqal Mcqu Loss,Entropy,Mean Token Accuracy,Num Tokens
0.581993,No log,796,0.156782,0.103464,0.944509,4779912.000000


Dropping fully masked examples from eval dataset: 100%|██████████| 931/931 [00:00<00:00, 33917.87 examples/s]


Training Loss,Validation Loss,Step,Sft Mediqal Mcqm Loss,Entropy,Mean Token Accuracy,Num Tokens
0.581993,No log,796,0.093300,0.077990,0.957314,4779912.000000


Dropping fully masked examples from eval dataset: 100%|██████████| 500/500 [00:00<00:00, 18074.53 examples/s]


Training Loss,Validation Loss,Step,Sft Ultramedical Loss,Entropy,Mean Token Accuracy,Num Tokens
0.581993,No log,796,0.784498,0.780872,0.759652,4779912.000000


,loss,token_accuracy,perplexity
val_before,0.778958,0.775677,2.179201
val_after,0.646693,0.801418,1.909217
test,0.592630,0.815342,1.808739
test_sft_mediqal_mcqu,0.156782,0.944509,1.169741
test_sft_mediqal_mcqm,0.093300,0.957314,1.097791
test_sft_ultramedical,0.784498,0.759652,2.191306


In [12]:
IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
STOP_IDS = [IM_END_ID, tokenizer.eos_token_id]
# Réponses mcqu : « C. texte de la proposition ». La lettre vient en premier : une réponse coupée reste évaluable.
# Le modèle de base, qui ne s'arrête pas toujours, est ainsi coupé plus tôt.
MAX_NEW_TOKENS = 64


def render(prompt):
    return tokenizer.apply_chat_template(prompt, add_generation_prompt=True, tokenize=False)


@torch.no_grad()
def generate_batch(prompts, batch_size=16, max_new_tokens=MAX_NEW_TOKENS, desc=None, model=None):
    # Génération par lots (GPU bien chargé) : renvoie les réponses, dans l'ordre des prompts
    model = model or trainer.model  # par défaut, le modèle du SFT
    texts = [render(prompt) for prompt in prompts]
    # Prompts triés par longueur : moins de padding dans chaque lot, les plus longs d'abord
    # (un manque de mémoire GPU apparaît dès le premier lot)
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]), reverse=True)
    answers = [None] * len(texts)
    padding_side, tokenizer.padding_side = tokenizer.padding_side, "left"  # padding à gauche : la génération suit chaque prompt
    try:
        for start in tqdm(range(0, len(order), batch_size), desc=desc, unit="lot"):
            batch = order[start:start + batch_size]
            inputs = tokenizer([texts[i] for i in batch], return_tensors="pt", padding=True, add_special_tokens=False).to(device)
            output = model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=False,
                eos_token_id=STOP_IDS, pad_token_id=tokenizer.pad_token_id,
            )
            decoded = tokenizer.batch_decode(output[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            for i, answer in zip(batch, decoded):
                answers[i] = answer.strip()
    finally:
        tokenizer.padding_side = padding_side
    return answers


def letters_accuracy(prediction, reference):
    # Accuracy : 1 si la réponse donne exactement les lettres attendues (« C. … » ou « - A. … » ligne par ligne)
    return float(answer_letters(prediction) == answer_letters(reference))


def letters_f1(prediction, reference):
    # F1 sur les lettres : crédit partiel quand une partie seulement des bonnes réponses est donnée (0 à 1)
    pred, ref = answer_letters(prediction), answer_letters(reference)
    common = len(pred & ref)
    if common == 0:
        return 0.0
    precision, recall = common / len(pred), common / len(ref)
    return 2 * precision * recall / (precision + recall)

In [13]:
N_EVAL = 20 if SMALL_RUN else 200    # exemples de test par source, pour accuracy et F1
BATCH_SIZE = 16                      # à baisser en cas de manque de mémoire GPU (OutOfMemoryError)

# Sources évaluées par génération, avec leur nombre maximal de tokens générés.
# Les réponses anglaises (plusieurs centaines de tokens, rédigées) ne se comparent pas mot à mot à une référence :
# elles sont évaluées par la loss et la perplexité (cellule précédente).
GENERATION_SOURCES = {"sft_mediqal_mcqu": MAX_NEW_TOKENS, "sft_mediqal_mcqm": 192}

trainer.model.eval()
trainer.model.config.use_cache = True  # accélère la génération

# 1. Qualité : génération par lots, modèle de base puis modèle SFT, source par source
source_generations = []
for source, max_new_tokens in GENERATION_SOURCES.items():
    subset = splits["test"].filter(lambda example: example["source"] == source)
    subset = subset.select(range(min(N_EVAL, len(subset))))
    prompts = list(subset["prompt"])
    with trainer.model.disable_adapter():  # modèle de base, sans LoRA
        base_answers = generate_batch(prompts, BATCH_SIZE, max_new_tokens, desc=f"Base {source}")
    sft_answers = generate_batch(prompts, BATCH_SIZE, max_new_tokens, desc=f"SFT {source}")
    source_generations.append(pd.DataFrame({
        "source": source,
        "question": [prompt[-1]["content"][-200:] for prompt in prompts],
        "reference": [completion[0]["content"] for completion in subset["completion"]],
        "base": base_answers,
        "sft": sft_answers,
    }))
generations = pd.concat(source_generations, ignore_index=True)
references = generations["reference"]
for model_name in ("base", "sft"):
    generations[f"accuracy_{model_name}"] = [letters_accuracy(a, r) for a, r in zip(generations[model_name], references)]
    generations[f"f1_{model_name}"] = [letters_f1(a, r) for a, r in zip(generations[model_name], references)]
generations.to_csv(FINAL_DIR / "test_generations.csv", index=False)

# Moyennes sur les exemples de test : modèle de base vs modèle SFT
QUALITY_METRICS = ["accuracy", "f1"]
comparison = pd.DataFrame({
    model_name: {metric: generations[f"{metric}_{model_name}"].mean() for metric in QUALITY_METRICS}
    for model_name in ("base", "sft")
})

# Qualité par source
quality_by_source = generations.groupby("source")[
    [f"{metric}_{model_name}" for metric in QUALITY_METRICS for model_name in ("base", "sft")]
].mean()
display(quality_by_source)
comparison

SFT sft_mediqal_mcqm: 100%|██████████| 13/13 [01:56<00:00,  8.94s/lot]


,accuracy_base,accuracy_sft,f1_base,f1_sft
source,,,,
sft_mediqal_mcqm,0.035,0.115,0.293036,0.586333
sft_mediqal_mcqu,0.005,0.320,0.023000,0.430833


,base,sft
accuracy,0.020000,0.217500
f1,0.158018,0.508583


In [14]:
# Quelques exemples à relire
pd.set_option("display.max_colwidth", 200)
generations.groupby("source")[["source", "question", "reference", "base", "sft"]].head(3)

,source,question,reference,base,sft
0,sft_mediqal_mcqu,potence fonctionnelle du membre supérieur\n\nLe délai habituel de consolidation d'une fracture de la clavicule est :\n\nPropositions :\nA. 1 semaine\nB. 2 semaines\nC. 4 semaines\nD. 8 semaines\nE...,C. 4 semaines,"Le délai habituel de consolidation d'une fracture de la clavicule est généralement de 4 semaines. Cependant, cela peut varier en fonction de plusieurs facteurs tels que l'âge, la santé générale, l...",C. 4 semaines
1,sft_mediqal_mcqu,"ycémie post-prandiale A 2 heures < 1,40 g/l.\nD. Glycémie A jeun < 0,95 g/l, glycémie post-prandiale A 2 heures < 1,20 g/l.\nE. Glycémie A jeun < 0,90 g/l, glycémie post-prandiale A 2 heures < 1,2...","D. Glycémie A jeun < 0,95 g/l, glycémie post-prandiale A 2 heures < 1,20 g/l.","La réponse correcte est : **C. Glycémie A jeun < 1,00 g/l, glycémie post-prandiale A 2 heures < 1,40 g/l.**\n\n### Explication :\nLe traitement du diabète gestationnel vise à mainten","C. Glycémie A jeun < 1,00 g/l, glycémie post-prandiale A 2 heures < 1,40 g/l."
2,sft_mediqal_mcqu,"tions :\nA. La notion de contage familial\nB. L'IDR à la tuberculine à 4mm\nC. La fièvre et les signes fonctionnels respiratoires\nD. Les propositions A et C exactes\nE. Les propositions A, B, C, ...",D. Les propositions A et C exactes,La proposition E est la plus appropriée. Les éléments suivants soutiennent l'étiologie tuberculeuse dans cette situation clinique :\n\n1. **La notion de contage familial** : L'enfant présente une ...,"E. Les propositions A, B, C, D exactes"
200,sft_mediqal_mcqm,ississement des muscles droits inférieurs\nC. Entraîne toujours une baisse d'acuité visuelle\nD. Impose un traitement d'urgence en cas de forme maligne\nE. Est toujours accompagnée d'un oedème pap...,- B. S'accompagne souvent d'un épaississement des muscles droits inférieurs\n- D. Impose un traitement d'urgence en cas de forme maligne,"La proposition D est incorrecte. L'exophthalmie basculienne n'implique pas nécessairement une baisse d'acuité visuelle. Elle peut être asymptomatique ou présenter des symptômes variés, tels que de...",- B. S'accompagne souvent d'un épaississement des muscles droits inférieurs\n- D. Impose un traitement d'urgence en cas de forme maligne
201,sft_mediqal_mcqm,?\n\nPropositions :\nA. Prélèvement de squames\nB. Examen en lumière de Wood\nC. Culture sur milieu de Sabouraud\nD. Recherche des anticorps spécifiques\nE. Recherche d'une hypersensibilité cutané...,- A. Prélèvement de squames\n- C. Culture sur milieu de Sabouraud,"Bonjour Monsieur <PERSON>, je suis un assistant médical et je vais vous aider à confirmer le diagnostic de lésion bilatérale prurigineuse inguinocrurale. Voici les étapes à suivre pour confirmer c...",- A. Prélèvement de squames\n- B. Examen en lumière de Wood
202,sft_mediqal_mcqm,en cas de paralysie du droit supérieur\nD. Il présente une rétraction lors de la survenue d'une maladie de Basedow\nE. La paralysie de sa commande nerveuse entraîne une ouverture de la fente palp...,- A. Il s'insère sur le bord du tarse de la paupière supérieure\n- B. Son fonctionnement est conservé en cas de paralysie faciale périphérique\n- D. Il présente une rétraction lors de la survenue ...,Le releveur de la paupière supérieure présente la ou les caractéristique(s) suivante(s) :\n\nPropositions :\nA. Il s'insère sur le bord du tarse de la paupière supérieure\nB. Son fonctionnement es...,- A. Il s'insère sur le bord du tarse de la paupière supérieure\n- B. Son fonctionnement est conservé en cas de paralysie faciale périphérique\n- C. Son fonctionnement est altéré en cas de paralys...


In [15]:
# Métriques finales dans MLflow (les loss sont déjà envoyées par le Trainer), puis clôture du run
mlflow.log_metrics({f"{name}_token_accuracy": row["token_accuracy"] for name, row in summary.iterrows()})
mlflow.log_metrics({f"{name}_perplexity": row["perplexity"] for name, row in summary.iterrows()})
mlflow.log_metrics({
    f"test_{metric}_{model}": comparison.loc[metric, model]
    for metric in QUALITY_METRICS
    for model in comparison.columns
})
# Par source : loss, token accuracy et perplexité du test, accuracy et F1 des réponses générées
mlflow.log_metrics({
    f"{name}_{metric}": value for name, row in per_source_summary.iterrows() for metric, value in row.items()
})
mlflow.log_metrics({
    f"test_{source}_{column}": value for source, row in quality_by_source.iterrows() for column, value in row.items()
})
mlflow.end_run()

🏃 View run sft-lora-r16-v2-seed42 at: https://dagshub.com/obiyohan/my-ml-mlflow.mlflow/#/experiments/2/runs/ff4f841e4d644be39b6cecae759fec58
🧪 View experiment at: https://dagshub.com/obiyohan/my-ml-mlflow.mlflow/#/experiments/2


## Lecture des résultats

| Métrique MLflow | Définition | Attendu |
|---|---|---|
| `loss`, `eval_loss` | courbes de loss d'entraînement et de validation | baissent ensemble ; si `eval_loss` remonte : surapprentissage |
| `val_before_loss`, `val_after_loss`, `test_loss` | loss avant / après SFT et sur le test | après < avant ; test proche de val |
| `*_token_accuracy` | part des tokens de la réponse correctement prédits | augmente après SFT |
| `*_perplexity` | exp(loss) | baisse après SFT |
| `test_sft_<source>_loss`, `_token_accuracy`, `_perplexity` | les mêmes mesures sur le test, source par source | chaque source s'améliore, y compris l'anglais |
| `test_accuracy_*` | réponse générée avec exactement les lettres attendues (sources françaises) | sft > base ; hasard ≈ 0,2 en `mcqu` |
| `test_f1_*` | F1 sur les lettres (crédit partiel pour les réponses multiples) | sft > base |
| `test_sft_<source>_{accuracy,f1}_*` | accuracy et F1, source par source | sft > base pour `mcqu` et `mcqm` |

- La courbe `eval_mean_token_accuracy` mélange toutes les évaluations (TRL ignore le préfixe) : utiliser `*_token_accuracy`.
- Accuracy et F1 portent sur les lettres : une réponse sans lettre (format non appris) compte 0.
- La latence (TTFT, TPOT, débit) sera mesurée en phase 3, sur le modèle final servi par vLLM.

**Suite** : passer `SMALL_RUN = False` pour l'entraînement complet, puis DPO (section 7).

## 7. Alignement DPO

Le DPO part du **modèle SFT** et lui apprend à préférer la réponse `chosen` à la réponse `rejected` de chaque paire.

- **Modèle de départ** : modèle de base + adaptateur SFT **fusionné** (les poids LoRA sont ajoutés aux poids du modèle). TRL calcule les log-probabilités directement depuis `lm_head`, ce qui impose de fusionner l'adaptateur SFT avant d'en ajouter un nouveau.
- **Nouvel adaptateur LoRA** entraîné par DPO. La **référence** du DPO est le modèle SFT lui-même : TRL l'obtient en désactivant l'adaptateur, sans charger un second modèle en mémoire.
- **Données** : paires françaises (MediQAl `mcqm`, vérifiées par le corrigé) et anglaises (UltraMedical, filtrées sur la qualité), mélangées pour l'entraînement, évaluées séparément par langue.
- **Reprise après redémarrage** : exécuter d'abord la section 1 (configuration), puis cette section. Le tokenizer et l'adaptateur SFT sont relus depuis `SFT_ADAPTER_DIR`. Avant la validation (7.4), exécuter aussi la cellule des fonctions de génération (section 6).

In [16]:
# Paires DPO de la phase 1, par langue
DPO_DATASETS = {"fr": "dpo_mediqal_mcqm", "en": "dpo_ultramedical"}
DPO_COLUMNS = ["source", "lang", "pair_check", "prompt", "chosen", "rejected"]  # colonnes communes aux deux jeux

# Hyperparamètres DPO
DPO_BETA = 0.1              # écart toléré avec la référence : plus beta est grand, plus le modèle reste proche du SFT
DPO_LEARNING_RATE = 5e-5    # LoRA : ~1 % des poids entraînés ; à 5e-6 (valeur usuelle en DPO complet), les marges restaient quasi nulles
DPO_MAX_LENGTH = 1024       # prompt + réponse, en tokens
DPO_N_EVAL = 20 if SMALL_RUN else 200  # exemples du test français pour la génération (7.4)
DPO_BATCH_SIZE = 16                    # génération par lots : à baisser en cas d'OutOfMemoryError

SFT_ADAPTER_DIR = Path("../outputs") / RUN_NAME / "final"  # adaptateur SFT de la section 5
DPO_RUN_NAME = f"dpo-lora-r16-{DATA_VERSION}-beta{DPO_BETA}-lr{DPO_LEARNING_RATE}-seed{SEED}" + ("-small" if SMALL_RUN else "")
DPO_OUTPUT_DIR = Path("../outputs") / DPO_RUN_NAME  # checkpoints, adaptateur DPO et modèle final fusionné
print(f"SFT : {SFT_ADAPTER_DIR}\nDPO : {DPO_OUTPUT_DIR}")

SFT : ../outputs/sft-lora-r16-v2-seed42/final
DPO : ../outputs/dpo-lora-r16-v2-beta0.1-lr5e-05-seed42


### 7.1 Données DPO

In [17]:
# Tokenizer sauvegardé avec l'adaptateur SFT : même gabarit qu'à l'entraînement SFT
tokenizer = AutoTokenizer.from_pretrained(SFT_ADAPTER_DIR)


def pair_length(example):
    # Longueur en tokens de la plus longue des deux conversations (prompt + chosen, prompt + rejected)
    return max(
        len(tokenizer(tokenizer.apply_chat_template(example["prompt"] + answer, tokenize=False), add_special_tokens=False)["input_ids"])
        for answer in (example["chosen"], example["rejected"])
    )


dpo_by_lang = {}
for lang, stem in DPO_DATASETS.items():
    files = {split: f"../data/anonymized/{stem}_{split}.anonymized.jsonl" for split in ("train", "validation", "test")}
    ds = load_dataset("json", data_files=files).select_columns(DPO_COLUMNS)
    # Paires trop longues : TRL couperait la fin de la réponse, on les écarte
    ds = ds.filter(lambda example: pair_length(example) <= DPO_MAX_LENGTH)
    if SMALL_RUN:
        for split, size in {"train": 32, "validation": 8, "test": 8}.items():
            ds[split] = ds[split].shuffle(seed=SEED).select(range(size))
    dpo_by_lang[lang] = ds
    print(f"{lang}: " + ", ".join(f"{split} {len(d)}" for split, d in ds.items()))

# Entraînement et validation : les deux langues mélangées
dpo_train = concatenate_datasets([ds["train"] for ds in dpo_by_lang.values()]).shuffle(seed=SEED)
dpo_validation = concatenate_datasets([ds["validation"] for ds in dpo_by_lang.values()]).shuffle(seed=SEED)
print(f"DPO : train {len(dpo_train)}, validation {len(dpo_validation)}")

Filter: 100%|██████████| 481/481 [00:00<00:00, 746.11 examples/s]


fr: train 1357, validation 220, test 481


Filter: 100%|██████████| 756/756 [00:02<00:00, 356.70 examples/s]

en: train 1994, validation 297, test 750
DPO : train 3351, validation 517


In [18]:
# Une paire telle que la verra le modèle
example = dpo_train[0]
print(tokenizer.apply_chat_template(example["prompt"], add_generation_prompt=True, tokenize=False))
print(f"--- CHOSEN :\n{example['chosen'][0]['content']}\n--- REJECTED :\n{example['rejected'][0]['content']}")

<|im_start|>system
You are an expert medical assistant. Answer accurately and clearly.<|im_end|>
<|im_start|>user
A person developed anterior wall MI and died within 2 hours of onset of the symptoms. The pathological autopsy of the hea and observation of the involved myocardium under light microscopy, is underway. What will the finding be, if any?

A. Beginning of coagulation necrosis
B. Neutrophilic infiltration
C. Waviness of myosite fibres at the border
D. Well developed phagocytosis of dead cells<|im_end|>
<|im_start|>assistant

--- CHOSEN :
Let's think step by step.

The patient died within 2 hours of onset of symptoms, which means the infarction is very acute. In the first 2 hours, there will be no visible changes on light microscopy. The earliest changes of myocardial infarction, such as coagulation necrosis, will not be apparent until around 4-6 hours after the onset of ischemia.

However, there will be some subtle changes that can be observed, particularly at the border of the

### 7.2 Modèle de départ : modèle SFT fusionné

In [19]:
# Libère la mémoire GPU du SFT si la section 5 vient de tourner
if "trainer" in globals():
    del trainer, model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

# Modèle de base + adaptateur SFT fusionné : point de départ et référence du DPO
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype)
sft_model = PeftModel.from_pretrained(base_model, str(SFT_ADAPTER_DIR)).merge_and_unload().to(device)
print(f"{sft_model.num_parameters() / 1e9:.2f} B paramètres")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1683.57it/s]


1.72 B paramètres


### 7.3 Entraînement DPO

- **LoRA** : même configuration que pour le SFT, sur un nouvel adaptateur.
- **Une époque** : le DPO sur-apprend vite. Au-delà, le modèle mémorise les paires sans mieux généraliser.
- **Batch effectif de 16** (2 × 8) : chaque exemple contient deux réponses, donc deux fois plus de tokens qu'en SFT.
- **Learning rate de 5e-5**, avec montée progressive sur 10 % des étapes. Plus élevé qu'en DPO complet (5e-6 à 5e-7) : avec LoRA, seule une petite partie des poids est entraînée. Au premier run, 5e-6 laissait la marge des récompenses quasi nulle sur le français et la courbe n'avait pas atteint son plateau. Si `logps/chosen` chute fortement, revenir à 2e-5.

In [20]:
dpo_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

# Suivi MLflow : même serveur que le SFT (connexion faite en section 5), expérience dédiée au DPO
if MLFLOW_BACKEND == "dagshub":
    import dagshub

    dagshub.init(repo_owner=DAGSHUB_REPO_OWNER, repo_name=DAGSHUB_REPO_NAME, mlflow=True)
else:
    mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("qwen3-1.7b-dpo-lora")
mlflow.start_run(run_name=DPO_RUN_NAME)
mlflow.log_params({
    "model": MODEL_NAME, "data_version": DATA_VERSION, "sft_adapter": str(SFT_ADAPTER_DIR), "small_run": SMALL_RUN, "device": device,
    "beta": DPO_BETA, "lora_r": dpo_lora_config.r, "lora_alpha": dpo_lora_config.lora_alpha,
    **{f"pairs_{lang}_{split}": len(d) for lang, ds in dpo_by_lang.items() for split, d in ds.items()},
})

dpo_args = DPOConfig(
    output_dir=str(DPO_OUTPUT_DIR),
    run_name=DPO_RUN_NAME,
    seed=SEED,
    # DPO
    beta=DPO_BETA,
    loss_type=["sigmoid"],  # perte DPO standard
    max_length=DPO_MAX_LENGTH,
    # Optimisation
    num_train_epochs=1,
    max_steps=10 if SMALL_RUN else -1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,  # batch effectif = 16 paires
    learning_rate=DPO_LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=0.1,  # 10 % des étapes en montée progressive du learning rate
    gradient_checkpointing=True,
    bf16=bf16,
    fp16=dtype == torch.float16,
    # Évaluation et checkpoints
    eval_strategy="steps",
    eval_steps=5 if SMALL_RUN else 25,
    save_steps=5 if SMALL_RUN else 25,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    # Logs
    logging_steps=1 if SMALL_RUN else 5,
    report_to="mlflow",
)

dpo_trainer = DPOTrainer(
    model=sft_model,
    args=dpo_args,
    train_dataset=dpo_train,
    eval_dataset=dpo_validation,
    processing_class=tokenizer,
    peft_config=dpo_lora_config,
)
dpo_trainer.model.print_trainable_parameters()

Initialized MLflow to track repo "obiyohan/my-ml-mlflow"

Repository obiyohan/my-ml-mlflow initialized!

Tokenizing train dataset: 100%|██████████| 3351/3351 [00:11<00:00, 298.64 examples/s]
Dropping fully truncated examples from train dataset: 100%|██████████| 3351/3351 [00:00<00:00, 35682.57 examples/s]
Tokenizing eval dataset: 100%|██████████| 517/517 [00:01<00:00, 263.82 examples/s]
Dropping fully truncated examples from eval dataset: 100%|██████████| 517/517 [00:00<00:00, 24822.06 examples/s]

trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


In [21]:
# Reprise après interruption : dpo_trainer.train(resume_from_checkpoint=True)
dpo_result = dpo_trainer.train()
dpo_result.metrics

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
25,0.609904,0.613660,0.660207,368517.000000,4.257034,4.316445,0.797700,-0.050707,-0.256157,0.701923,0.205449,-161.677670,-151.615092
50,0.568255,0.539519,0.742001,736179.000000,4.244421,4.386052,0.788465,-1.080856,-2.334070,0.761538,1.253214,-171.979156,-172.394231
75,0.539723,0.495345,0.684158,1099978.000000,3.601760,3.744082,0.790372,-1.000988,-2.351082,0.759615,1.350094,-171.180479,-172.564347
100,0.427735,0.459467,0.709782,1454806.000000,3.443772,3.605047,0.788535,-1.291662,-2.765911,0.771154,1.474249,-174.087214,-176.712633
125,0.401555,0.445329,0.724640,1822311.000000,3.566563,3.736736,0.786570,-1.551480,-3.092520,0.776923,1.541041,-176.685391,-179.978728
150,0.414196,0.429755,0.717473,2191027.000000,3.488621,3.663503,0.786645,-1.560295,-3.246097,0.778846,1.685801,-176.773548,-181.514491
175,0.366943,0.434244,0.720000,2560031.000000,3.432321,3.609744,0.785683,-1.756260,-3.555399,0.776923,1.799139,-178.733198,-184.607518
200,0.455742,0.427334,0.721698,2939074.000000,3.447248,3.624027,0.785359,-1.745821,-3.548738,0.788462,1.802917,-178.628807,-184.540911
210,0.532386,0.432038,0.721658,3079731.000000,3.447972,3.626215,0.785750,-1.752957,-3.543772,0.786538,1.790814,-178.700167,-184.491242


{'train_runtime': 3144.9608,
 'train_samples_per_second': 1.066,
 'train_steps_per_second': 0.067,
 'total_flos': 3.3865276404547584e+16,
 'train_loss': 0.4752013791175116,
 'epoch': 1.0}

In [22]:
# Sauvegarde de l'adaptateur DPO (meilleur checkpoint) et de la configuration du run
DPO_FINAL_DIR = DPO_OUTPUT_DIR / "final"
dpo_trainer.save_model(str(DPO_FINAL_DIR))
tokenizer.save_pretrained(str(DPO_FINAL_DIR))

dpo_run_config = {
    "model": MODEL_NAME,
    "sft_adapter": str(SFT_ADAPTER_DIR),  # l'adaptateur DPO s'applique au modèle SFT fusionné
    "seed": SEED,
    "small_run": SMALL_RUN,
    "beta": DPO_BETA,
    "lora": {"r": dpo_lora_config.r, "alpha": dpo_lora_config.lora_alpha, "dropout": dpo_lora_config.lora_dropout},
    "data": {f"{lang}_{split}": len(d) for lang, ds in dpo_by_lang.items() for split, d in ds.items()},
    "training_args": dpo_args.to_dict(),
}
(DPO_FINAL_DIR / "run_config.json").write_text(json.dumps(dpo_run_config, indent=2, default=str), encoding="utf-8")

4897

### 7.4 Validation DPO

1. **Préférences** sur la validation et le test de chaque langue, et sur le test anglais séparé selon la longueur (`chosen` plus longue ou plus courte que la `rejected`) : une précision très différente entre les deux révèle une préférence apprise pour la longueur plutôt que pour le fond. La *récompense implicite* d'une réponse mesure combien le modèle DPO la juge plus probable que le modèle SFT. On en tire :
   - `reward_accuracy` : part des paires où la récompense de la `chosen` dépasse celle de la `rejected` (0,5 = hasard) ;
   - `reward_margin` : écart moyen entre les deux récompenses ;
   - `logps_chosen` / `logps_rejected` : log-probabilités moyennes des deux réponses sous le modèle DPO.
2. **Réponses générées** sur le test français : modèle SFT vs modèle DPO, avec l'accuracy et le F1 sur les lettres de la section 6. Le test `mcqm` a un corrigé : on compare chaque réponse à la liste des bonnes réponses.

In [23]:
# 1. Préférences, par split et par langue
# (TRL nomme ses métriques « eval_rewards/... » quel que soit le préfixe : on les renomme ici)
eval_sets = {f"{split}_{lang}": ds[split] for lang, ds in dpo_by_lang.items() for split in ("validation", "test")}

# Test anglais selon la longueur : la chosen est-elle plus longue ou plus courte que la rejected ?
test_en = dpo_by_lang["en"]["test"]
chosen_longer = [len(c[0]["content"]) > len(r[0]["content"]) for c, r in zip(test_en["chosen"], test_en["rejected"])]
eval_sets["test_en_chosen_longer"] = test_en.select([i for i, longer in enumerate(chosen_longer) if longer])
eval_sets["test_en_chosen_shorter"] = test_en.select([i for i, longer in enumerate(chosen_longer) if not longer])

dpo_rows = {}
for name, dataset in eval_sets.items():
    if len(dataset) == 0:  # petit run : un sous-ensemble peut être vide
        continue
    m = dpo_trainer.evaluate(dataset, metric_key_prefix=name)
    dpo_rows[name] = {
        "pairs": len(dataset),
        "loss": m[f"{name}_loss"],
        "reward_accuracy": m["eval_rewards/accuracies"],
        "reward_margin": m["eval_rewards/margins"],
        "logps_chosen": m["eval_logps/chosen"],
        "logps_rejected": m["eval_logps/rejected"],
    }
dpo_summary = pd.DataFrame(dpo_rows).T
dpo_summary

Tokenizing eval dataset: 100%|██████████| 220/220 [00:00<00:00, 420.97 examples/s]
Dropping fully truncated examples from eval dataset: 100%|██████████| 220/220 [00:00<00:00, 27433.31 examples/s]


Training Loss,Validation Loss,Step,Fr Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
0.532386,No log,210,0.516982,0.020546,3079731.000000,-1.982683,-1.832580,0.954914,-0.935684,-1.796565,0.718182,0.860881,-12.841273,-22.850674


Tokenizing eval dataset: 100%|██████████| 481/481 [00:01<00:00, 427.88 examples/s]
Dropping fully truncated examples from eval dataset: 100%|██████████| 481/481 [00:00<00:00, 36379.48 examples/s]


Training Loss,Validation Loss,Step,Fr Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
0.532386,No log,210,0.521143,0.022599,3079731.000000,-1.911168,-1.776739,0.950426,-0.954689,-1.862925,0.739669,0.908236,-12.968380,-23.635552


Tokenizing eval dataset: 100%|██████████| 297/297 [00:01<00:00, 245.84 examples/s]
Dropping fully truncated examples from eval dataset: 100%|██████████| 297/297 [00:00<00:00, 15613.71 examples/s]


Training Loss,Validation Loss,Step,En Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
0.532386,No log,210,0.361302,0.826044,3079731.000000,4.269321,4.402315,0.758456,-2.374937,-4.893806,0.846667,2.518869,-302.190576,-304.985557


... (3 outputs left out)

Training Loss,Validation Loss,Step,En Chosen Longer Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
0.532386,No log,210,0.768130,0.849119,3079731.000000,4.449820,4.357706,0.736811,-2.469131,-3.037644,0.635496,0.568513,-396.890650,-251.536058


Tokenizing eval dataset: 100%|██████████| 226/226 [00:00<00:00, 226.74 examples/s]
Dropping fully truncated examples from eval dataset: 100%|██████████| 226/226 [00:00<00:00, 9346.68 examples/s]


Training Loss,Validation Loss,Step,En Chosen Shorter Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
0.532386,No log,210,0.560202,0.808293,3079731.000000,4.348353,4.359563,0.759825,-2.621409,-4.443798,0.754386,1.822389,-296.035246,-342.625883


,pairs,loss,reward_accuracy,reward_margin,logps_chosen,logps_rejected
validation_fr,220.0,0.516982,0.718182,0.860881,-12.841273,-22.850674
test_fr,481.0,0.521143,0.739669,0.908236,-12.968380,-23.635552
validation_en,297.0,0.361302,0.846667,2.518869,-302.190576,-304.985557
test_en,750.0,0.705156,0.668883,0.941667,-366.188085,-278.668982
test_en_chosen_longer,524.0,0.768130,0.635496,0.568513,-396.890650,-251.536058
test_en_chosen_shorter,226.0,0.560202,0.754386,1.822389,-296.035246,-342.625883


In [24]:
# 2. Réponses générées sur le test français : modèle SFT (adaptateur DPO désactivé) vs modèle DPO
assert "generate_batch" in globals(), "Exécuter d'abord la cellule des fonctions de génération (section 6)"
DPO_MAX_NEW_TOKENS = 192  # réponses mcqm : listes de 2 à 5 options

dpo_model = dpo_trainer.model.eval()
dpo_model.config.use_cache = True  # accélère la génération

test_fr = dpo_by_lang["fr"]["test"].select(range(min(DPO_N_EVAL, len(dpo_by_lang["fr"]["test"]))))
fr_prompts = list(test_fr["prompt"])
fr_references = [chosen[0]["content"] for chosen in test_fr["chosen"]]

with dpo_model.disable_adapter():
    fr_sft_answers = generate_batch(fr_prompts, DPO_BATCH_SIZE, DPO_MAX_NEW_TOKENS, desc="Génération SFT", model=dpo_model)
fr_dpo_answers = generate_batch(fr_prompts, DPO_BATCH_SIZE, DPO_MAX_NEW_TOKENS, desc="Génération DPO", model=dpo_model)

dpo_generations = pd.DataFrame({
    "question": [prompt[-1]["content"][-200:] for prompt in fr_prompts],
    "reference": fr_references,
    "sft": fr_sft_answers,
    "dpo": fr_dpo_answers,
})
for model_name in ("sft", "dpo"):
    dpo_generations[f"accuracy_{model_name}"] = [letters_accuracy(a, r) for a, r in zip(dpo_generations[model_name], fr_references)]
    dpo_generations[f"f1_{model_name}"] = [letters_f1(a, r) for a, r in zip(dpo_generations[model_name], fr_references)]
dpo_generations.to_csv(DPO_FINAL_DIR / "test_fr_generations.csv", index=False)

DPO_QUALITY_METRICS = ["accuracy", "f1"]
dpo_comparison = pd.DataFrame({
    model_name: {metric: dpo_generations[f"{metric}_{model_name}"].mean() for metric in DPO_QUALITY_METRICS}
    for model_name in ("sft", "dpo")
})
dpo_comparison

Génération DPO: 100%|██████████| 13/13 [01:58<00:00,  9.13s/lot]


,sft,dpo
accuracy,0.085000,0.165000
f1,0.586813,0.633488


In [25]:
# Quelques exemples à relire
dpo_generations[["question", "reference", "sft", "dpo"]].head(5)

,question,reference,sft,dpo
0,u diurétique thiazidique du furosémide (Lasilix®)\nD. Ne pas augmenter les apports hydriques quand on prescrit un diurétique thiazidique\nE. Ne pas prescrire un régime sans sel strict chez les gen...,- D. Ne pas augmenter les apports hydriques quand on prescrit un diurétique thiazidique\n- E. Ne pas prescrire un régime sans sel strict chez les gens âgés,- A. Commencer par de faibles doses de diurétiques thiazidiques\n- B. Utiliser une association de diurétiques contenant un épargneur de potassium,- A. Commencer par de faibles doses de diurétiques thiazidiques\n- B. Utiliser une association de diurétiques contenant un épargneur de potassium
1,smodique à l'origine de cet angor ?\n\nPropositions :\nA. Crises exclusivement spontanées\nB. Crises nocturnes à horaire fixe\nC. Crises de longue durée\nD. Evolution périodique\nE. Epreuve d'effo...,- A. Crises exclusivement spontanées\n- B. Crises nocturnes à horaire fixe\n- D. Evolution périodique,- A. Crises exclusivement spontanées\n- B. Crises nocturnes à horaire fixe\n- D. Evolution périodique,- B. Crises nocturnes à horaire fixe\n- D. Evolution périodique
2,peut s'accompagner de la(des) manifestation(s) clinique(s) suivante(s) :\n\nPropositions :\nA. Crises convulsives\nB. Troubles de la conscience\nC. Douleurs abdominales\nD. Prurit cutané intense\n...,- B. Troubles de la conscience\n- C. Douleurs abdominales\n- D. Prurit cutané intense\n- E. Polyurie,- A. Crises convulsives\n- B. Troubles de la conscience\n- C. Douleurs abdominales,- A. Crises convulsives\n- C. Douleurs abdominales
3,la glène interne\nC. Aspect effilé de l'épine tibiale interne\nD. Deux microgéodes sous-chondrales au niveau du condyle interne\nE. Les axes diaphysaires du fémur et du tibia forment un varus de 5...,"- A. Pincement d'environ 50 %, localisé a l'interligne fémoro-tibiale interne\n- B. Densification sous-chondrale de la glène interne\n- C. Aspect effilé de l'épine tibiale interne\n- D. Deux micro...","- A. Pincement d'environ 50 %, localisé a l'interligne fémoro-tibiale interne\n- B. Densification sous-chondrale de la glène interne","- A. Pincement d'environ 50 %, localisé a l'interligne fémoro-tibiale interne\n- D. Deux microgéodes sous-chondrales au niveau du condyle interne"
4,e cette lésion ?\n\nPropositions :\nA. Prolifération épithéliale\nB. Présence de ponts d'union\nC. Présence de globes cornés\nD. Envahissement du derme superficiel\nE. Embols tumoraux des vaisseau...,- A. Prolifération épithéliale\n- D. Envahissement du derme superficiel,- A. Prolifération épithéliale\n- C. Présence de globes cornés,- B. Présence de ponts d'union\n- C. Présence de globes cornés


In [26]:
# Métriques finales dans MLflow, puis clôture du run
mlflow.log_metrics({
    f"{name}_{metric}": value
    for name, row in dpo_summary.iterrows()
    for metric, value in row.items()
})
mlflow.log_metrics({
    f"test_fr_{metric}_{model}": dpo_comparison.loc[metric, model]
    for metric in DPO_QUALITY_METRICS
    for model in dpo_comparison.columns
})
mlflow.end_run()

🏃 View run dpo-lora-r16-v2-beta0.1-lr5e-05-seed42 at: https://dagshub.com/obiyohan/my-ml-mlflow.mlflow/#/experiments/3/runs/64d7a1a405b84345b0a98ffa8f4a688a
🧪 View experiment at: https://dagshub.com/obiyohan/my-ml-mlflow.mlflow/#/experiments/3


### 7.5 Modèle final fusionné

Modèle de base + SFT + DPO en un seul jeu de poids complets (environ 3,4 Go en bf16), avec le tokenizer et son gabarit : c'est le modèle à servir avec vLLM en phase 3.

In [27]:
MERGED_DIR = DPO_OUTPUT_DIR / "merged"
merged_model = dpo_trainer.model.merge_and_unload()
merged_model.save_pretrained(str(MERGED_DIR))
tokenizer.save_pretrained(str(MERGED_DIR))
print(f"Modèle final : {MERGED_DIR}")

Writing model shards: 100%|██████████| 1/1 [00:09<00:00,  9.06s/it]

Modèle final : ../outputs/dpo-lora-r16-v2-beta0.1-lr5e-05-seed42/merged


## Lecture des résultats DPO

| Métrique MLflow | Définition | Attendu |
|---|---|---|
| `loss`, `eval_loss` | perte DPO d'entraînement et de validation (0,69 = aucune préférence apprise) | baissent sous 0,69 ; si `eval_loss` remonte : surapprentissage |
| `rewards/accuracies` | part des paires où la `chosen` l'emporte sur la `rejected` | monte nettement au-dessus de 0,5 |
| `rewards/margins` | écart moyen de récompense entre `chosen` et `rejected` | augmente, sans exploser |
| `logps/chosen`, `logps/rejected` | log-probabilités moyennes des deux réponses | `rejected` baisse plus que `chosen` ; si `chosen` chute fortement, baisser le learning rate ou augmenter beta |
| `{validation,test}_{fr,en}_*` | les mêmes métriques, par split et par langue | proches entre validation et test, et entre langues |
| `test_en_chosen_{longer,shorter}_*` | test anglais, selon que la `chosen` est la plus longue ou la plus courte | précisions proches ; un grand écart = préférence pour la longueur |
| `test_fr_accuracy_*`, `test_fr_f1_*` | lettres des réponses générées sur le test `mcqm`, comparées au corrigé | dpo ≥ sft |

- Les courbes `eval_rewards/*` mélangent toutes les évaluations (TRL ignore le préfixe) : pour la comparaison finale, utiliser `{validation,test}_{fr,en}_*`.
- **Itérer** : un run par jeu d'hyperparamètres (`DPO_BETA`, `DPO_LEARNING_RATE`), avec le même `SEED`. Le nom du run les contient.

**Suite** : contrôles de sécurité (hallucinations, recommandations dangereuses) sur le modèle final, puis déploiement avec vLLM (phase 3).